In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# End-to-End Agent Evaluation: Intent Detection & Gemini 2.5 Grading

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fdfcx-scrapi%2Fmain%2Fexamples%2Fvertex_ai_conversation%2Fend_to_end_agent_evaluation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/dfcx-scrapi/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb">
      <img width="32px" src="https://www.svgrepo.com/download/217753/github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/dfcx-scrapi/blob/main/examples/vertex_ai_conversation/end_to_end_agent_evaluation.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>


| Author |
| --- |
| [Aniket Agrawal](https://github.com/aniketagrawal2012) |

## Overview

This notebook consolidates the entire evaluation workflow into a single script. It connects to a live Dialogflow CX agent to generate responses and then immediately uses **Gemini 2.5 Flash** to grade those responses against a Ground Truth.

### Workflow
1.  **Robust Data Loading**: Load a CSV of test queries, handling formatting edge cases (missing columns, casing).
2.  **Intent Detection (DFCX)**: Send queries to the agent, capturing the `Response Text`, `Matched Intent`, and `Latency`.
3.  **Generative Evaluation (Gemini 2.5)**: Use `gemini-2.5-flash` as an evaluator to score the response (1-5) on **Accuracy**, **Completeness**, **Relevance**, and **Clarity**.
4.  **Reporting**: Export a final comprehensive report.



**Language Support:** This workflow supports multilingual agents (e.g., **Chinese zh-tw**) effectively due to Gemini's cross-lingual reasoning capabilities.

## Prerequisites
- Google Cloud Project with **Dialogflow API** and **Vertex AI API** enabled.
- An existing Dialogflow CX Agent.
- A CSV file containing test queries and expected answers.

In [ ]:
# Install necessary libraries
!pip install google-cloud-dialogflow-cx google-cloud-aiplatform pandas openpyxl --quiet

## 1. Setup & Authentication

In [ ]:
import sys
import IPython
import time
import uuid
import pandas as pd
import re
import vertexai

# Import DFCX Client
from google.cloud.dialogflowcx_v3beta1.services.sessions import SessionsClient
from google.cloud.dialogflowcx_v3beta1.types import session

# Import Vertex AI Generative Model
from vertexai.generative_models import GenerativeModel, GenerationConfig

# Authenticate User
if "google.colab" in sys.modules:
    from google.colab import auth
    from google.auth import default
    auth.authenticate_user()
    credentials, _ = default()

# Restart kernel to ensure libraries are loaded
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

## 2. Configuration

Define your environment variables. We explicitly use `gemini-2.5-flash` for the evaluation model.

In [ ]:
# --- Project Details ---
PROJECT_ID = ""  # @param {type:"string"}
LOCATION = ""    # @param {type:"string"} (e.g., us-central1)

# --- Agent Details ---
AGENT_ID = ""    # @param {type:"string"}
LOCATION_AGENT = "" # @param {type:"string"} (e.g., us, global)
LANGUAGE_CODE = "en" # @param {type:"string"} (e.g., en, zh-tw)

# --- Evaluation Model ---
EVAL_MODEL_NAME = "gemini-2.5-flash"

# --- Input Data ---
INPUT_CSV_FILE = "" # @param {type:"string"}

# Initialize Vertex AI
if PROJECT_ID:
    vertexai.init(project=PROJECT_ID, location=LOCATION)
    # Initialize the Evaluator Model
    eval_model = GenerativeModel(
        model_name=EVAL_MODEL_NAME,
        generation_config={"temperature": 0.1} # Low temp for consistent grading
    )

## 3. Robust Data Loading

We use `transform_csv_dataframe` to normalize the input CSV. This improves robustness by handling missing columns, defaults, and casing inconsistencies.

In [ ]:
def transform_csv_dataframe(csv_df):
    """Transforms a raw CSV DataFrame to the standardized evaluation format.
    
    Args:
        csv_df: A pandas DataFrame loaded from a CSV file.

    Returns:
        A normalized DataFrame with columns: [eval_id, action_id, action_type, action_input, ...]
    """
    # Create an empty DataFrame with the required columns schema
    transformed_df = pd.DataFrame(columns=[
        'eval_id', 'action_id', 'action_type', 'action_input',
        'action_input_parameters', 'tool_action', 'notes'
    ])

    for index, row in csv_df.iterrows():
        # Robustness: Handle missing columns using .get() defaults
        eval_id = row.get('eval_id', f"conv_{index}")
        action_id = int(row.get('action_id', index))
        action_input_parameters = row.get('action_input_parameters', '')
        raw_type = str(row.get('action_type', '')).strip()

        # Normalize action_type casing
        if raw_type.lower() == 'user utterance':
            action_type = 'User Utterance'
        elif raw_type.lower() == 'agent response':
            action_type = 'Agent Response'
        elif raw_type.lower() == 'tool invocation':
            action_type = 'Tool Invocation'
        else:
            action_type = raw_type

        # Append normalized row
        transformed_df.loc[index] = [
            eval_id, action_id, action_type, row.get('action_input', ''),
            action_input_parameters, row.get('tool_action', ''), row.get('notes', '')
        ]

    return transformed_df

def preprocess_data(df):
    """Extracts User Inputs (Questions) and Expected Outputs (Ground Truth)."""
    user_utterances = []
    ground_truths = []

    for index, row in df.iterrows():
        if row['action_type'] == 'User Utterance':
            user_utterances.append(row['action_input'])
        elif row['action_type'] == 'Agent Response':
            ground_truths.append(row['action_input'])

    return user_utterances, ground_truths

# Load Data
if INPUT_CSV_FILE:
    try:
        raw_df = pd.read_csv(INPUT_CSV_FILE)
        df_processed = transform_csv_dataframe(raw_df)
        user_utterances, ground_truths = preprocess_data(df_processed)
        print(f"Loaded {len(user_utterances)} test queries.")
    except Exception as e:
        print(f"Error loading file: {e}")

## 4. Intent Detection Function (DFCX)

Calls the Dialogflow CX API to get the real-time response.

In [ ]:
def detect_intent(project_id, location, agent_id, text, language_code):
    """Sends query to Dialogflow CX and returns the response text and matched intent."""
    client_options = None
    if location != "global":
        api_endpoint = f"{location}-dialogflow.googleapis.com:443"
        client_options = {"api_endpoint": api_endpoint}

    session_client = SessionsClient(client_options=client_options)
    session_id = str(uuid.uuid4())
    
    session_path = session_client.session_path(
        project=project_id, location=location, agent=agent_id, session=session_id
    )
    
    text_input = session.TextInput(text=text)
    query_input = session.QueryInput(text=text_input, language_code=language_code)
    
    request = session.DetectIntentRequest(
        session=session_path, query_input=query_input
    )

    try:
        response = session_client.detect_intent(request=request)
        # Attempt to get text response
        if response.query_result.response_messages:
            response_text = response.query_result.response_messages[0].text.text[-1]
        else:
            response_text = "NO_TEXT_RESPONSE"
            
        # Matched Intent Name (DisplayName)
        intent_name = response.query_result.intent.display_name
        
    except Exception as e:
        print(f"Interaction Error: {e}")
        response_text = "ERROR"
        intent_name = "ERROR"

    return response_text, intent_name

## 5. Evaluation Function (Gemini 2.5)

Configures `gemini-2.5-flash` to act as a judge. We use a **complete 5-point rubric** to ensure granular evaluation.

In [ ]:
def evaluate_aspect(aspect_name, questions, ground_truths, obtained_answers, model):
    """Evaluates a specific quality aspect using Gemini 2.5 Flash."""
    
    eval_responses = []
    scores = []

    # Detailed definitions for prompt
    criteria_map = {
        "accuracy": "Does the obtained answer provide factually correct information that aligns with the ground truth?",
        "completeness": "Does the obtained answer cover all essential details present in the ground truth?",
        "relevance": "Does the obtained answer directly address the question?",
        "clarity_coherence": "Is the obtained answer well-organized, easy to understand, and free of grammatical errors?"
    }
    
    # COMPLETE Scoring Guidelines (1-5)
    score_guide = {
        1: "The obtained answer is completely irrelevant, incorrect, or unintelligible.",
        2: "The obtained answer has significant omissions, inaccuracies, or is hard to follow.",
        3: "The obtained answer is mostly correct but lacks specific details or clarity found in the ground truth.",
        4: "The obtained answer is very good, accurate and clear, with only minor deviations.",
        5: "The obtained answer is perfect, comprehensive, and matches the ground truth in quality."
    }

    prompt_template = """
    Task: Evaluate the similarity between a ground truth answer and an obtained answer for the given question regarding {aspect_name}.

    Inputs:
    * Question: {question}
    * Ground truth answer: {ground_truth}
    * Obtained answer: {obtained_answer}

    Criteria: {criteria}

    Scoring Rubric:
    1: {s1}
    2: {s2}
    3: {s3}
    4: {s4}
    5: {s5}

    Output:
    Provide a score (1-5) (e.g. **Score: 3**) followed by a brief reasoning.
    """

    print(f"--- Evaluating Aspect: {aspect_name} ---")
    
    for i in range(len(questions)):
        prompt = prompt_template.format(
            aspect_name=aspect_name,
            question=questions[i],
            ground_truth=ground_truths[i],
            obtained_answer=obtained_answers[i],
            criteria=criteria_map.get(aspect_name, "Evaluate quality"),
            s1=score_guide[1], 
            s2=score_guide[2], 
            s3=score_guide[3], 
            s4=score_guide[4], 
            s5=score_guide[5]
        )

        try:
            # Generate content using Gemini 2.5
            resp = model.generate_content(prompt).text
        except ValueError:
            resp = "Score: 1 (Blocked by Safety Filter)"
        
        eval_responses.append(resp)
        
        # Extract Score
        match = re.search(r"Score\s*:\s*(\d+)|Score:\s*\**(\d+)\**", resp)
        if match:
            # Returns the first non-None group
            scores.append(int(match.group(1) or match.group(2)))
        else:
            scores.append(None)
        
        if i % 5 == 0:
            print(f"Evaluated {i} records.")

    return eval_responses, scores

## 6. Main Execution Loop

1. Loop through queries.
2. Call DFCX (Detect).
3. Calculate Latency.
4. Call Gemini 2.5 (Evaluate) for all aspects.

In [ ]:
results = []

print(f"Starting process for {len(user_utterances)} records...")

limit = min(len(user_utterances), len(ground_truths))

# 1. BATCH INTENT DETECTION
obtained_answers = []
intents = []
latencies = []

for i in range(limit):
    start_time = time.time()
    response, intent = detect_intent(
        PROJECT_ID, LOCATION_AGENT, AGENT_ID, user_utterances[i], LANGUAGE_CODE
    )
    latency = time.time() - start_time
    
    obtained_answers.append(response)
    intents.append(intent)
    latencies.append(latency)
    
    if i % 5 == 0:
        print(f"[DFCX] Processed {i} queries. Latency: {latency:.2f}s")

# 2. BATCH EVALUATION (Gemini 2.5)
# We evaluate 'Accuracy' and 'Relevance' as key metrics
_, score_acc = evaluate_aspect("accuracy", user_utterances[:limit], ground_truths[:limit], obtained_answers, eval_model)
_, score_rel = evaluate_aspect("relevance", user_utterances[:limit], ground_truths[:limit], obtained_answers, eval_model)

# 3. Aggregate
for i in range(limit):
    results.append({
        "question": user_utterances[i],
        "ground_truth": ground_truths[i],
        "obtained_answer": obtained_answers[i],
        "matched_intent": intents[i],
        "latency": round(latencies[i], 3),
        "score_accuracy": score_acc[i],
        "score_relevance": score_rel[i]
    })

## 7. Export & Summary

Save the combined report.

In [ ]:
df_results = pd.DataFrame(results)

print("\n--- Evaluation Summary ---")
print(f"Avg Latency: {df_results['latency'].mean():.4f}s")
print(f"Avg Accuracy: {df_results['score_accuracy'].mean():.2f}")
print(f"Avg Relevance: {df_results['score_relevance'].mean():.2f}")

output_csv = "final_evaluation_report.csv"
df_results.to_csv(output_csv, index=False)
print(f"Saved detailed report to {output_csv}")
display(df_results.head())